## Rancangan Eksperimen

### Konsep singkat
- **Backbone (model pretrained):** model yang sudah "membaca" jutaan kalimat sehingga paham pola bahasa.
  Ibaratnya orang yang sudah fasih berbahasa, tapi belum pernah diminta menilai sentimen.
- **Head:** lapisan kecil di ujung model yang mengubah representasi kalimat menjadi 3 skor kelas.
  Head bawaan model dibuang dan diganti head baru dengan 3 output (*head replacement*).
- **Fine-tuning:** melatih ulang model pada data kita supaya cocok dengan tugas sentimen.
- **Freezing:** membekukan sebagian parameter (`requires_grad = False`) sehingga tidak ikut diperbarui.
  Lebih sedikit yang dilatih berarti lebih hemat waktu dan memori, tetapi model kurang bisa beradaptasi.
- **F1 makro vs akurasi:** data didominasi kelas positif (58%). Model yang selalu menebak "positif"
  pun sudah dapat akurasi 58%. F1 makro menghitung F1 tiap kelas lalu merata-ratakannya, sehingga
  kelas kecil (netral, 10%) dihitung sama pentingnya.
- **Seed dan simpangan baku antar-seed:** seed mengatur keacakan (inisialisasi head, urutan data).
  Seed berbeda bisa memberi hasil sedikit berbeda. Dengan 3 seed, kita bisa melihat seberapa besar
  variasi "kebetulan" itu, lalu membedakannya dari selisih yang benar-benar nyata.

### Hipotesis
- **H1 (backbone):** IndoBERT mencapai F1 makro tertinggi dibanding mBERT dan DistilBERT pada strategi
  yang sama, karena korpus pretraining dan kosakatanya selaras dengan Bahasa Indonesia.
  XLM-R diperkirakan bersaing dengan IndoBERT, dengan biaya parameter dan inferensi lebih besar.
- **H2 (strategi):** full > partial > probing pada F1 makro. Partial kehilangan sedikit F1 tetapi
  lebih hemat; probing paling hemat tetapi F1-nya jauh lebih rendah.
- **H3 (efisiensi):** DistilBERT sekitar 40% lebih kecil dan inferensinya 1,6–2× lebih cepat
  dibanding mBERT, dengan penurunan F1 makro yang kecil.

### Matriks eksperimen (24 run)
| Backbone \ Strategi | Full | Partial | Probing |
|---|---|---|---|
| mBERT | ✓ ×3 | ✓ ×3 | — |
| DistilBERT | ✓ ×3 | ✓ ×3 | — |
| IndoBERT | ✓ ×3 | ✓ ×3 | ✓ ×3 |
| XLM-R | ✓ ×3 | — | — |

### Variabel yang di gunakan
| Variabel | Nilai | Alasan |
|---|---|---|
| `max_length` | 128 | 99% kalimat ≤ 79 kata, jadi 128 token cukup tanpa banyak terpotong |
| Batch size | 32 | Ukuran umum untuk fine-tuning model base |
| Epoch | 3 | Rentang umum 2–4 epoch untuk fine-tuning |
| Seed | 42, 123, 2024 | Sama untuk semua run agar adil |
| Optimizer | AdamW, weight decay 0,01 | Standar fine-tuning Transformer |
| Scheduler | Linear, warmup 10% | LR naik perlahan di awal agar bobot pretrained tidak "kaget" |
| LR full/partial | 2e-5 | Kecil, karena bobot pretrained hanya perlu disesuaikan sedikit |
| LR probing | 1e-3 | Besar, karena hanya head baru (dari nol) yang dilatih |
| Presisi | fp16 bila GPU | Hemat memori dan waktu |  

### Alasan desain
- Label data uji SmSA disembunyikan, jadi **validasi resmi (yang dibersihkan) dipakai sebagai held-out**.
  Supaya tetap jujur, **tidak ada early stopping maupun pemilihan checkpoint** berdasarkan validasi.
  Setiap run dilatih dengan epoch tetap, lalu dievaluasi satu kali.
- Semua 24 run dijalankan dalam **satu sesi** di GPU yang sama, karena waktu inferensi hanya adil
  dibandingkan dalam kondisi yang sama.
- **Aturan menyimpulkan:** dua konfigurasi dianggap berbeda meyakinkan hanya jika selisih rata-ratanya
  lebih besar daripada simpangan baku antar-seed.

In [1]:
import os
import sys

# Paksa 1 GPU. Di Kaggle (T4 x2) ini mencegah DataParallel agar waktu sebanding.
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false" # Mematikan peringatan 

import csv
import gc
import hashlib
import json
import platform
import time
from datetime import datetime
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
import torch
import transformers
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    set_seed,
)

# cek GPU aja 
def cek_perangkat():
    if torch.cuda.is_available():
        properti_gpu = torch.cuda.get_device_properties(0)
        return {
            "device": "cuda",
            "gpu_name": properti_gpu.name,
            "vram_gb": properti_gpu.total_memory / (1024 ** 3),
        }
    return {"device": "cpu", "gpu_name": "-", "vram_gb": 0.0}


# folder utk menyimpan output dlm csv dan grafik nantinya
def tentukan_folder_output(is_kaggle):
    if is_kaggle:
        return Path("/kaggle/working")
    folder_sekarang = Path.cwd()
    # Jupyter menjalankan notebook dari folder notebooks/, jadi naik satu tingkat ke root repo.
    if folder_sekarang.name == "notebooks":
        return folder_sekarang.parent / "outputs"
    return folder_sekarang / "outputs"


# spill spek tipis-tipis
info_perangkat = cek_perangkat()
DEVICE = info_perangkat["device"]
GPU_NAME = info_perangkat["gpu_name"]
VRAM_GB = info_perangkat["vram_gb"]
USE_FP16 = DEVICE == "cuda"       
IS_KAGGLE = Path("/kaggle/working").exists()

# Folder output
OUTPUT_DIR = tentukan_folder_output(IS_KAGGLE)
FIGURES_DIR = OUTPUT_DIR / "figures"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# Ringkasan
print("Environment")
print(f"  Platform     : {'Kaggle' if IS_KAGGLE else 'Lokal'}")
print(f"  Device       : {DEVICE}")
print(f"  GPU          : {GPU_NAME}")
print(f"  VRAM         : {VRAM_GB:.1f} GB")
print(f"  fp16         : {USE_FP16}")
if DEVICE == "cpu":
    print("GPU tidak terdeteksi.")

print("\nVERSI Library: ")
print(f"  Python       : {platform.python_version()}")
print(f"  torch        : {torch.__version__} (CUDA {torch.version.cuda})")
print(f"  transformers : {transformers.__version__}")
print(f"  scikit-learn : {sklearn.__version__}")
print(f"  pandas       : {pd.__version__}")
print(f"  numpy        : {np.__version__}")
print(f"  matplotlib   : {matplotlib.__version__}")

print("\nFOLDER OUTPUT")
print(f"  CSV    : {OUTPUT_DIR}")
print(f"  Grafik : {FIGURES_DIR}")

/home/sembiring/env/pytorch/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Environment
  Platform     : Lokal
  Device       : cuda
  GPU          : NVIDIA GeForce RTX 4050 Laptop GPU
  VRAM         : 5.6 GB
  fp16         : True

VERSI Library: 
  Python       : 3.14.4
  torch        : 2.13.0+cu132 (CUDA 13.2)
  transformers : 5.22.2
  scikit-learn : 1.9.0
  pandas       : 3.0.5
  numpy        : 2.5.2
  matplotlib   : 3.11.1

FOLDER OUTPUT
  CSV    : /home/sembiring/machine_learning/00_pr0j3ct/indo-sentiment-finetuning-benchmark/outputs
  Grafik : /home/sembiring/machine_learning/00_pr0j3ct/indo-sentiment-finetuning-benchmark/outputs/figures


In [2]:
# 3. KONFIGURASI

# ==== sedikit catatan utk menjalankan MODE ==== 
# "uji"   : cek kode dengan cepat (subset kecil, 1 epoch, 1 seed). Untuk laptop.
# "penuh" : eksperimen resmi 24 run dengan variabel terkendali di bawah. Untuk Kaggle.
MODE = "uji"

# utk MODE="uji"
UJI_BACKBONE = ["indobert"]
UJI_JUMLAH_DATA_LATIH = 500   # cukup untuk mengecek kode, bukan untuk hasil
UJI_EPOCH = 1
UJI_BATCH_SIZE = 8            # kecil agar aman di vram

# Variabel terkendali (mode penuh)
MAX_LENGTH = 128
BATCH_SIZE = 32
EPOCH = 3
SEEDS = [42, 123, 2024]
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1            # 10% dari total step dipakai untuk warmup
LR_PER_STRATEGI = {
    "full": 2e-5,
    "partial": 2e-5,
    "probing": 1e-3,          # lebih besar karena hanya head baru yang dilatih
}

# biaya waktu
INFER_JUMLAH_KALIMAT = 200    # sampel tetap dari validasi
INFER_JUMLAH_WARMUP = 10      # beberapa prediksi awal tidak dihitung (GPU masih "pemanasan")

# Backbone dan matriks eksperimen 
BACKBONES = {
    "mbert": "bert-base-multilingual-cased",
    "distilbert": "distilbert-base-multilingual-cased",
    "indobert": "indobenchmark/indobert-base-p1",
    "xlmr": "xlm-roberta-base",
}

# Strategi yang dijalankan untuk tiap backbone (sesuai matriks 24 run).
MATRIKS_EKSPERIMEN = {
    "mbert": ["full", "partial"],
    "distilbert": ["full", "partial"],
    "indobert": ["full", "partial", "probing"],
    "xlmr": ["full"],
}

In [3]:
# 4. DAFTAR RUN DAN KODE KONFIGURASI

# Mengubah UJI_BACKBONE menjadi list kode backbone
def daftar_backbone_uji():
    if UJI_BACKBONE == "semua":
        return list(MATRIKS_EKSPERIMEN.keys())
    if isinstance(UJI_BACKBONE, str):
        return [UJI_BACKBONE]
    return list(UJI_BACKBONE)

# hentikan proses kalau isi konfigurasi ada yg salah
def validasi_konfigurasi():
    assert MODE in ("uji", "penuh"), f'MODE harus "uji" atau "penuh", bukan {MODE!r}'
    for backbone, daftar_strategi in MATRIKS_EKSPERIMEN.items():
        assert backbone in BACKBONES, f"Backbone '{backbone}' tidak ada di BACKBONES"
        for strategi in daftar_strategi:
            assert strategi in LR_PER_STRATEGI, f"Strategi '{strategi}' tidak punya learning rate"
    if MODE == "uji":
        for backbone in daftar_backbone_uji():
            assert backbone in MATRIKS_EKSPERIMEN, (
                f"UJI_BACKBONE berisi '{backbone}', pilihan yang ada: {list(MATRIKS_EKSPERIMEN)}"
            )
# Mengembalikan pengaturan yang berlaku untuk mode tertentu
def ambil_pengaturan_mode(mode):
    if mode == "penuh":
        return {
            "mode": "penuh",
            "backbones": list(MATRIKS_EKSPERIMEN.keys()),
            "seeds": SEEDS,
            "epoch": EPOCH,
            "batch_size": BATCH_SIZE,
            "subset_latih": None,          # None = pakai seluruh data latih
        }
    return {
        "mode": "uji",
        "backbones": daftar_backbone_uji(),
        "seeds": [SEEDS[0]],               # cukup 1 seed untuk mengecek kode
        "epoch": UJI_EPOCH,
        "batch_size": UJI_BATCH_SIZE,
        "subset_latih": UJI_JUMLAH_DATA_LATIH,
    }


def susun_daftar_run(pengaturan):
    daftar_run = []
    for backbone in pengaturan["backbones"]:
        for strategi in MATRIKS_EKSPERIMEN[backbone]:
            for seed in pengaturan["seeds"]:
                run = {
                    "run_id": f"{backbone}_{strategi}_s{seed}",
                    "backbone": backbone,
                    "checkpoint": BACKBONES[backbone],
                    "strategy": strategi,
                    "seed": seed,
                    "lr": LR_PER_STRATEGI[strategi],
                    "max_length": MAX_LENGTH,
                    "batch_size": pengaturan["batch_size"],
                    "epoch": pengaturan["epoch"],
                    "subset_latih": pengaturan["subset_latih"],
                    "mode": pengaturan["mode"],
                }
                daftar_run.append(run)
    return daftar_run

# Menghitung kode konfigurasi = 8 karakter pertama
def hitung_kode_konfigurasi(daftar_run):
    isi_konfigurasi = {
        "runs": daftar_run,
        "weight_decay": WEIGHT_DECAY,
        "warmup_ratio": WARMUP_RATIO,
        "infer_jumlah_kalimat": INFER_JUMLAH_KALIMAT,
        "infer_jumlah_warmup": INFER_JUMLAH_WARMUP,
    }
    teks_json = json.dumps(isi_konfigurasi, sort_keys=True)
    return hashlib.md5(teks_json.encode("utf-8")).hexdigest()[:8]


# Validasi konfigurasi
validasi_konfigurasi()

PENGATURAN = ambil_pengaturan_mode(MODE)
DAFTAR_RUN = susun_daftar_run(PENGATURAN)
KODE = hitung_kode_konfigurasi(DAFTAR_RUN)

if MODE == "penuh":
    assert len(DAFTAR_RUN) == 24, f"Mode penuh seharusnya 24 run, tetapi tersusun {len(DAFTAR_RUN)}"

HASIL_CSV = OUTPUT_DIR / f"hasil_smsa_{KODE}.csv"
REKAP_CSV = OUTPUT_DIR / f"rekap_agregat_smsa_{KODE}.csv"

print("=" * 60)
print("RINGKASAN KONFIGURASI")
print("=" * 60)
print(f"  MODE          : {MODE}")
print(f"  Backbone      : {PENGATURAN['backbones']}")
print(f"  Seed          : {PENGATURAN['seeds']}")
print(f"  Epoch         : {PENGATURAN['epoch']}")
print(f"  Batch size    : {PENGATURAN['batch_size']}")
print(f"  Data latih    : {'penuh' if PENGATURAN['subset_latih'] is None else PENGATURAN['subset_latih']}")
print(f"  Jumlah run    : {len(DAFTAR_RUN)}")
print(f"  Kode          : {KODE}")
print(f"  CSV hasil     : {HASIL_CSV}")
print(f"  CSV rekap     : {REKAP_CSV}")

kolom_tampil = ["run_id", "checkpoint", "strategy", "seed", "lr", "batch_size", "epoch"]
display(pd.DataFrame(DAFTAR_RUN)[kolom_tampil])

RINGKASAN KONFIGURASI
  MODE          : uji
  Backbone      : ['indobert']
  Seed          : [42]
  Epoch         : 1
  Batch size    : 8
  Data latih    : 500
  Jumlah run    : 3
  Kode          : f29f97ca
  CSV hasil     : /home/sembiring/machine_learning/00_pr0j3ct/indo-sentiment-finetuning-benchmark/outputs/hasil_smsa_f29f97ca.csv
  CSV rekap     : /home/sembiring/machine_learning/00_pr0j3ct/indo-sentiment-finetuning-benchmark/outputs/rekap_agregat_smsa_f29f97ca.csv


,run_id,checkpoint,strategy,seed,lr,batch_size,epoch
0,indobert_full_s42,indobenchmark/indobert-base-p1,full,42,0.00002,8,1
1,indobert_partial_s42,indobenchmark/indobert-base-p1,partial,42,0.00002,8,1
2,indobert_probing_s42,indobenchmark/indobert-base-p1,probing,42,0.00100,8,1
